# input and output definitions

> What goes into a comparison, and what comes out of one.

This is the first of four notebooks that make up `estravon_bench`
(`00_input_and_output_def` → `01_client` → `02_report` → `03_compare`). The
package is authored with [nbdev](https://nbdev.fast.ai/): the code cells
below are exported by `nbdev_export` into `estravon_bench/io.py`, not
written there directly. Editing that `.py` file directly has no lasting
effect -- the next export overwrites it.

Cells marked `#| export` ship in the generated module. Cells marked
`#| hide` run during `nbdev_test` but are excluded from the module and from
the rendered docs -- mostly tests and exploration. The cells are
executable; running one, editing a value, and re-running is the normal way
to read this notebook.

This notebook has two sections: what a comparison needs as input (a PDF to
test against, below) and what it produces as output (`ComparisonResult`,
further down). Both export to the same file, `estravon_bench/io.py`, named
for that split.

Directive reference:
- `#| default_exp io` (next cell) sets this notebook's export target:
  `estravon_bench/io.py`.
- `#| export` marks a cell as shipped code.
- `#| hide` marks a cell excluded from the module and the docs.
- `#| eval: false` marks a cell that is shown but not executed by
  `nbdev_test` or the docs build -- used below for the cell that triggers a
  real download.
- Editing a notebook does not update the `.py` file by itself -- run
  `nbdev_export` afterward (or the notebook's last cell, which calls it).

In [ ]:
#| default_exp io

In [ ]:
#| hide
from nbdev.showdoc import *

## Data in: a real PDF to test against

`compare()` needs an actual PDF to run engines against. This package ships
no PDF of its own -- committing a multi-page book into a git repository is
the wrong way to distribute it, and a benchmark should run against a real
document, not a synthetic one built to make every engine look good.

The fixture used here is *La Scienza in Cucina e l'Arte di Mangiar Bene*
(Pellegrino Artusi, 1891), from the Internet Archive
(identifier `artusi-1891`, public domain). The item has two PDF variants of
the same book:

| Variant | Size | Text layer |
|---|---|---|
| `Artusi 1891_text.pdf` | ~90 MB | Yes (OCR'd) |
| `Artusi 1891.pdf` | ~293 MB | No (raw scan) |

That split happens to line up with one of the axes this benchmark cares
about: whether a source PDF already has a text layer changes how several
engines behave (Datalab/Marker in particular reads an existing text layer
directly and only falls back to OCR when one is absent). `get_artusi()`
defaults to the smaller, text-layer variant; `get_artusi(scanned=True)`
fetches the larger scanned-only one.

Downloading is handled by [`fastdownload`](https://fastdownload.fast.ai/), a
small companion library to `fastcore` (already a dependency via `nbdev`).
`FastDownload.download(url)` saves to a local cache directory and returns
the cached path on every call after the first -- nothing re-downloads
unless the cached file fails its recorded checksum. Checksums are stored in
`estravon_bench/download_checks.py`, written by `FastDownload.update(url)`
and committed to the repository, so a corrupted or unexpectedly changed
download is caught rather than silently used.

In [ ]:
#| export
from __future__ import annotations

from pathlib import Path

from fastdownload import FastDownload

import estravon_bench as _pkg

_ARTUSI_TEXT_URL = "https://archive.org/download/artusi-1891/Artusi%201891_text.pdf"
_ARTUSI_SCAN_URL = "https://archive.org/download/artusi-1891/Artusi%201891.pdf"

_dl = FastDownload(module=_pkg, base="~/.cache/estravon_bench")


def get_artusi(scanned: bool = False) -> Path:
    """Return a local path to the Artusi (1891) cookbook PDF, downloading and
    caching it on first call.

    ``scanned=False`` (default): the ~90 MB OCR'd, text-layer variant.
    ``scanned=True``: the ~293 MB raw-scan, no-text-layer variant.

    Public domain, from the Internet Archive (`artusi-1891`). The checksum
    for whichever variant has been fetched at least once via
    ``FastDownload.update()`` is recorded in ``download_checks.py`` and
    verified on every subsequent call.
    """
    url = _ARTUSI_SCAN_URL if scanned else _ARTUSI_TEXT_URL
    return _dl.download(url)

### Try it

The cell below is marked `#| eval: false` -- it is real, working code, but
it downloads ~90 MB on first run, so it does not execute automatically as
part of `nbdev_test` or the docs build. We run it directly whenever we
actually want the file: `get_artusi()` returns a `pathlib.Path` to the
cached PDF, ready to pass into `compare()`.

In [ ]:
#| eval: false
path = get_artusi()
print(path, path.stat().st_size)

Rendering a page to check the file downloaded correctly needs a PDF
rasteriser, which `estravon_bench` does not otherwise depend on --
[PyMuPDF](https://pymupdf.readthedocs.io/) is used here only, not added to
`pyproject.toml`. Install it separately (`pip install pymupdf`) to run this
cell.

In [ ]:
#| eval: false
import fitz
from IPython.display import Image

doc = fitz.open(path)
print(f"{doc.page_count} pages")
pix = doc[0].get_pixmap(dpi=100)
Image(data=pix.tobytes())

## Data out: `ComparisonResult`

`compare()` (in `03_compare`) runs one PDF through several engines and
returns one `ComparisonResult` per engine. `client.py` builds it from an
HTTP response, `report.py` renders lists of it into a table, `compare.py`
produces it.

Two fields need more context than their type gives:
- `error`: set when an engine failed. `compare()` records this instead of
  raising, so a failed engine shows up as a row with `error` set rather than
  stopping the run. `.ok` returns `error is None`.
- `local` / `cost_usd`: `cost_usd=None` means the cost was not determined.
  `cost_usd=0.0, local=True` means the engine runs on the caller's own
  machine and has no per-call dollar cost (MinerU today). The two states are
  kept separate so a rendered table can distinguish "free, local compute"
  from "unknown."

In [ ]:
#| export
from dataclasses import dataclass


@dataclass
class ComparisonResult:
    """One engine's outcome for one compare() call.

    ``local=True`` means the engine has no per-call dollar cost (MinerU
    today) -- ``cost_usd`` is ``0.0`` in that case, never a fabricated
    figure. ``error`` is set (and every other field left at its default)
    when this engine failed or was unreachable -- compare() never raises
    on a single engine's failure, it records a row instead.
    """

    engine: str
    markdown: str | None = None
    predict_time_s: float | None = None
    cost_usd: float | None = None
    local: bool = False
    page_count: int | None = None
    backend_url: str | None = None
    error: str | None = None

    @property
    def ok(self) -> bool:
        return self.error is None

### Try it

The cell below builds one successful and one failed `ComparisonResult` and
checks `.ok`, `.local`, and `.markdown` on each. It runs as part of
`nbdev_test`.

In [ ]:
#| hide
r = ComparisonResult(engine="mineru", markdown="# Hi", cost_usd=0.0, local=True, page_count=3)
assert r.ok
assert r.local is True
print(r)

e = ComparisonResult(engine="mistral", error="connection refused")
assert not e.ok
assert e.markdown is None
print(e)

---
Next: [`01_client`](01_client.ipynb) -- the HTTP client that builds
`ComparisonResult`s from a real `estravon-backend` response, plus the
subprocess orchestration that spins up a local instance per engine.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()